# Песочница паттернов GoF: Strategy, Factory Method, Observer
**Исполнитель:** Киселев И. Д., группа ГИ-41, вариант 11 «Путевое хозяйство».

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java. Каждый пример представлен **одним файлом** `.java` и запускается без отдельной компиляции командой `java Файл.java` (запуск исходного файла, Java 11 и выше).

**Идея задания.** Паттерн изучается не по описанию, а по разнице: сначала запускается код «до» — работающий, но жёстко связанный; затем тот же сценарий переписывается на паттерн, и проверки доказывают, что **поведение не изменилось**, а **новое требование добавляется без правки старого кода**.

**Состав работы**

| Раздел | Паттерн | Что делает студент |
|---|---|---|
| 2 | Strategy | **образец** — запущен и разобран |
| 3 | Factory Method | задача 1: дефекты рельсов, написано на варианте 11 |
| 4 | Observer | задача 2: оповещение о дефекте, написано на варианте 11 |
| 5–6 | — | сравнение, выводы, контрольные вопросы |

Международные идентификаторы кода — латиницей и описательно, комментарии — по-русски. Каждый файл сам себя проверяет: печатает строки `[OK]` / `[FAIL]` и бросает `AssertionError`, если хотя бы одна проверка не пройдена.

**Сдача.** Три файла `.java`, три диаграммы `.puml`, блокнот и вывод ячейки самопроверки — в pull request.

## 1. Подготовка среды

In [1]:
# 1.1 Проверить, какая Java доступна в среде.
# Нужна версия 11 или выше: только с неё работает запуск одного исходного файла (java Файл.java).
!java -version
!javac -version

Если версия ниже 11, запуск одного исходного файла недоступен — всюду в блокноте вместо `!java Файл.java` используйте два шага: `!javac Файл.java && java Файл`.

Проверка «живого» запуска (ячейка 1.2 записывает файл, 1.3 — запускает):

In [2]:
%%writefile Hello.java
public class Hello {
    public static void main(String[] args) {
        System.out.println("Среда готова, Java работает: " + System.getProperty("java.version"));
    }
}

Writing Hello.java


In [3]:
!java Hello.java

Среда готова, Java работает: 21.0.13


---

## 2. Образец: Strategy (разобран полностью)

**Сценарий.** Расчёт платы за перевозку груза. Плата зависит от вида отправки: повагонная, контейнерная, маршрутная. Формулы учебные.

### 2.1 Код «до»: один метод знает про все виды отправок

In [4]:
%%writefile TariffBefore.java
// Код «до»: вид отправки разбирается условным оператором внутри одного метода.
public class TariffBefore {

    public static void main(String[] args) {
        System.out.printf("повагонная   : %10.2f%n", cost("WAGON", 60, 500));
        System.out.printf("контейнерная : %10.2f%n", cost("CONTAINER", 24, 500));
        System.out.printf("маршрутная   : %10.2f%n", cost("BLOCK_TRAIN", 3000, 500));
        System.out.printf("рефрижератор : %10.2f%n", cost("REFRIGERATED", 60, 500));
    }

    static double cost(String shipmentType, double massTons, int distanceKm) {
        double base;
        if (shipmentType.equals("WAGON")) {
            base = 1.20 * massTons * distanceKm / 100.0 + 150.0;
        } else if (shipmentType.equals("CONTAINER")) {
            base = 0.90 * massTons * distanceKm / 100.0 + 400.0;
        } else if (shipmentType.equals("BLOCK_TRAIN")) {
            base = 0.70 * massTons * distanceKm / 100.0;
        } else {
            base = 0.0;   // про новый вид отправки метод не знает
        }
        return Math.round(base * 100.0) / 100.0;
    }
}

Writing TariffBefore.java


In [5]:
!java TariffBefore.java

повагонная   :     510,00
контейнерная :     508,00
маршрутная   :   10500,00
рефрижератор :       0,00


### 2.2 Разбор: что именно плохо

Запустив код «до», обратите внимание:

* рефрижераторная отправка дала **0.00** — метод о ней не знает, и единственный способ это исправить — залезть внутрь уже работающего метода;
* каждый новый вид отправки удлиняет один и тот же метод, то есть нарушает **принцип открытости/закрытости** (открыт для расширения, закрыт для изменения);
* метод одновременно отвечает за выбор варианта и за сам расчёт — нарушение **единственной ответственности**;
* протестировать один вид отправки отдельно от остальных нельзя.

**Что делает Strategy.** Каждый вариант расчёта становится отдельным классом с общим интерфейсом, а вызывающий код работает с интерфейсом и больше не знает перечня вариантов.

### 2.3 Код «после»: вариант расчёта вынесен в стратегию

In [6]:
%%writefile TariffAfter.java
// Код «после»: каждый вид отправки — отдельная стратегия расчёта.
import java.util.LinkedHashMap;
import java.util.Map;

public class TariffAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {

        // Каталог стратегий: вызывающий код собирает его, но не содержит формул
        Map<String, TariffStrategy> tariffCatalog = new LinkedHashMap<>();
        tariffCatalog.put("WAGON", new WagonTariff());
        tariffCatalog.put("CONTAINER", new ContainerTariff());
        tariffCatalog.put("BLOCK_TRAIN", new BlockTrainTariff());

        TariffCalculator tariffCalculator = new TariffCalculator();

        System.out.println("--- Расчёт по каталогу стратегий (масса 60 т, расстояние 500 км) ---");
        for (Map.Entry<String, TariffStrategy> catalogEntry : tariffCatalog.entrySet()) {
            double calculatedCost = tariffCalculator.calculate(catalogEntry.getValue(), 60, 500);
            System.out.printf("%-12s %-28s %10.2f%n",
                    catalogEntry.getKey(), catalogEntry.getValue().title(), calculatedCost);
        }

        System.out.println();
        System.out.println("--- Характеризующие тесты: поведение совпадает с кодом «до» ---");
        check("повагонная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("WAGON"), 60, 500),
                           legacyCost("WAGON", 60, 500)));
        check("контейнерная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("CONTAINER"), 24, 500),
                           legacyCost("CONTAINER", 24, 500)));
        check("маршрутная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("BLOCK_TRAIN"), 3000, 500),
                           legacyCost("BLOCK_TRAIN", 3000, 500)));

        System.out.println();
        System.out.println("--- Тест открытости/закрытости: новый вид отправки без правки существующих классов ---");
        TariffStrategy refrigeratedTariff = new TariffStrategy() {
            public double cost(double massTons, int distanceKm) {
                return 1.80 * massTons * distanceKm / 100.0 + 600.0;
            }
            public String title() {
                return "рефрижераторная отправка";
            }
        };
        double refrigeratedCost = tariffCalculator.calculate(refrigeratedTariff, 60, 500);
        System.out.printf("%-12s %-28s %10.2f%n", "REFRIGERATED", refrigeratedTariff.title(), refrigeratedCost);
        check("новая стратегия рассчитана, класс TariffCalculator не изменён", refrigeratedCost > 0.0);
        check("в коде «до» тот же вид отправки давал нуль", legacyCost("REFRIGERATED", 60, 500) == 0.0);

        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }

    // --- служебное ---

    static boolean equalMoney(double firstValue, double secondValue) {
        return Math.abs(firstValue - secondValue) < 0.01;
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }

    // Старая реализация сохранена только как эталон поведения для характеризующих тестов
    static double legacyCost(String shipmentType, double massTons, int distanceKm) {
        double base;
        if (shipmentType.equals("WAGON")) {
            base = 1.20 * massTons * distanceKm / 100.0 + 150.0;
        } else if (shipmentType.equals("CONTAINER")) {
            base = 0.90 * massTons * distanceKm / 100.0 + 400.0;
        } else if (shipmentType.equals("BLOCK_TRAIN")) {
            base = 0.70 * massTons * distanceKm / 100.0;
        } else {
            base = 0.0;
        }
        return Math.round(base * 100.0) / 100.0;
    }
}

// --- участники паттерна Strategy ---

/** Стратегия: общий интерфейс всех вариантов расчёта. */
interface TariffStrategy {
    double cost(double massTons, int distanceKm);
    String title();
}

/** Контекст: умеет считать, но не знает, какие бывают виды отправок. */
class TariffCalculator {
    double calculate(TariffStrategy tariffStrategy, double massTons, int distanceKm) {
        double calculatedValue = tariffStrategy.cost(massTons, distanceKm);
        return Math.round(calculatedValue * 100.0) / 100.0;
    }
}

class WagonTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 1.20 * massTons * distanceKm / 100.0 + 150.0;
    }
    public String title() {
        return "повагонная отправка";
    }
}

class ContainerTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 0.90 * massTons * distanceKm / 100.0 + 400.0;
    }
    public String title() {
        return "контейнерная отправка";
    }
}

class BlockTrainTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 0.70 * massTons * distanceKm / 100.0;
    }
    public String title() {
        return "маршрутная отправка";
    }
}

Writing TariffAfter.java


In [7]:
!java TariffAfter.java

--- Расчёт по каталогу стратегий (масса 60 т, расстояние 500 км) ---
WAGON        повагонная отправка              510,00
CONTAINER    контейнерная отправка            670,00
BLOCK_TRAIN  маршрутная отправка              210,00

--- Характеризующие тесты: поведение совпадает с кодом «до» ---
[OK]   повагонная отправка
[OK]   контейнерная отправка
[OK]   маршрутная отправка

--- Тест открытости/закрытости: новый вид отправки без правки существующих классов ---
REFRIGERATED рефрижераторная отправка        1140,00
[OK]   новая стратегия рассчитана, класс TariffCalculator не изменён
[OK]   в коде «до» тот же вид отправки давал нуль

Все проверки пройдены.


### 2.4 Диаграмма классов образца

Установка PlantUML занимает около минуты и выполняется один раз за сеанс.

In [8]:
!apt-get -qq install -y plantuml > /dev/null && echo "PlantUML установлен"

In [9]:
%%writefile strategy.puml
@startuml
title Strategy: расчёт платы за перевозку (образец)

interface TariffStrategy {
  +cost(massTons, distanceKm) : double
  +title() : String
}

class TariffCalculator {
  +calculate(tariffStrategy, massTons, distanceKm) : double
}

class WagonTariff
class ContainerTariff
class BlockTrainTariff

class "Анонимный класс\n(рефрижераторная отправка)" as RefrigeratedTariff

TariffCalculator o--> TariffStrategy : использует
TariffStrategy <|.. WagonTariff
TariffStrategy <|.. ContainerTariff
TariffStrategy <|.. BlockTrainTariff
TariffStrategy <|.. RefrigeratedTariff

note right of RefrigeratedTariff
  Добавлен в main без правки
  TariffCalculator и других стратегий
end note
@enduml

Writing strategy.puml


In [10]:
!plantuml -tpng strategy.puml
from IPython.display import Image
Image("strategy.png")

### 2.5 Вопросы к образцу (ответы)

1. **Какие классы пришлось бы изменить в коде «до» и в коде «после», чтобы добавить рефрижераторную отправку?**
   **Ответ:** в коде «до» пришлось бы править метод `TariffBefore.cost`: добавить ещё одну ветку `else if` с формулой, то есть изменить уже работающий и протестированный класс. В коде «после» не меняется ни один существующий класс расчёта: добавляется новый класс, реализующий `TariffStrategy` (в образце — анонимный класс прямо в `main`), и регистрируется в каталоге. `TariffCalculator` остаётся неизменным; меняется только конфигурация каталога в точке сборки приложения.

2. **Зачем в файле оставлен метод `legacyCost`, если он больше не используется в расчёте?**
   **Ответ:** это эталон старого поведения для characterize-тестов (характеризующих проверок). С его помощью проверяется, что три прежних вида отправки считаются после рефакторинга точно так же, а новый вид в старой реализации давал `0.00`. То есть метод не участвует в производном расчёте, а служит тестовым оракулом и наглядным доказательством, что рефакторинг не изменил результат. В производственном коде такой метод был бы удалён вместе с тестами миграции.

3. **Что произойдёт, если в каталоге не окажется запрошенного вида отправки, и как это следует обработать?**
   **Ответ:** `tariffCatalog.get(...)` вернёт `null`, и при вызове `tariffStrategy.cost(...)` возникнет `NullPointerException` без внятного сообщения. Следует явно проверять наличие ключа (`containsKey` / `Objects.requireNonNull` с сообщением) и либо бросать `IllegalArgumentException` с указанием неизвестного типа, либо подставлять стратегию по умолчанию; главное — fail fast и понятная диагностика, а не случайный `null` в глубине расчёта.

---

## 3. Задача 1: Factory Method

**Сценарий.** Описание вагона по его типу: крытый, платформа, цистерна. Ниже — код «до», построенный на той же ошибке: условный оператор вместо полиморфизма.

### 3.1 Код «до» (запустить, не изменяя)

In [11]:
%%writefile FactoryBefore.java
// Код «до»: создание и описание объекта подменены разбором строкового типа.
public class FactoryBefore {

    public static void main(String[] args) {
        System.out.println(describe("COVERED", 68));
        System.out.println(describe("PLATFORM", 71));
        System.out.println(describe("TANK", 66));
        System.out.println(describe("HOPPER", 70));   // новый тип — обработчика нет
    }

    static String describe(String wagonType, double capacityTons) {
        if (wagonType.equals("COVERED")) {
            return "Крытый вагон: грузоподъёмность " + capacityTons + " т, тарные и штучные грузы";
        } else if (wagonType.equals("PLATFORM")) {
            return "Платформа: грузоподъёмность " + capacityTons + " т, длинномерные и тяжеловесные грузы";
        } else if (wagonType.equals("TANK")) {
            return "Цистерна: грузоподъёмность " + capacityTons + " т, наливные грузы";
        }
        return "Неизвестный тип вагона: " + wagonType;
    }
}

Writing FactoryBefore.java


In [12]:
!java FactoryBefore.java

### 3.2 Задание

Переписать сценарий на **Factory Method** в файле `FactoryAfter.java`. Требования:

1. Общий тип продукта: интерфейс или абстрактный класс `Wagon` с методами `description()` и `capacityTons()`.
2. Не менее трёх конкретных продуктов: `CoveredWagon`, `Platform`, `TankWagon`.
3. Создатель с фабричным методом: абстрактный класс или интерфейс `WagonCreator` с методом `createWagon(double capacityTons)` и конкретные создатели для каждого типа.
4. Вызывающий код (`main`) работает **только** с типами `WagonCreator` и `Wagon` и не содержит ни одного `if` по типу вагона.
5. Четвёртый тип (`HopperWagon`, зерновоз) добавляется **новым классом**, без правки ранее написанных классов; это должно быть видно по коду.
6. Проверки в файле (метод `check`, как в образце), минимум четыре:
   * описание каждого из трёх типов не пустое и содержит грузоподъёмность;
   * тип созданного объекта соответствует ожидаемому создателю;
   * четвёртый тип работает наравне с остальными;
   * метод `main` не содержит разбора типов (проверяется глазами преподавателя, в код выносить не нужно).
7. При неуспешной проверке — `throw new AssertionError(...)`.

**Подсказка по структуре файла:** первым идёт `public class FactoryAfter` с `main`, после него — интерфейсы и классы паттерна (как в `TariffAfter.java`).

In [13]:
%%writefile FactoryAfter.java
// Задача 1, вариант 11 «Путевое хозяйство»: Factory Method для дефектов рельсов.
// Клиентский код работает только с абстракциями RailDefectFactory и RailDefect.
import java.util.Arrays;
import java.util.List;

public class FactoryAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {

        // Каталог создателей. Четвёртый создатель добавлен отдельным классом:
        // для этого не потребовалось править RailDefectFactory или первые три создателя.
        List<RailDefectFactory> defectFactories = Arrays.asList(
                new HeadCrackFactory(),
                new WebCrackFactory(),
                new FootCorrosionFactory(),
                new GaugeWideningFactory());

        String[] expectedCodes = {"ГТ", "ТШ", "КП", "УК"};
        String[] expectedClassNames = {
                "HeadCrackDefect", "WebCrackDefect", "FootCorrosionDefect", "GaugeWideningDefect"
        };

        System.out.println("--- Дефекты рельсов, созданные фабричным методом ---");
        for (RailDefectFactory defectFactory : defectFactories) {
            RailDefect defect = defectFactory.createDefect(128.5, 45);
            System.out.printf("%-14s %-30s код=%-3s балл=%d  ограничение=%d км/ч%n",
                    defectFactory.getClass().getSimpleName(),
                    defect.title(), defect.code(), defect.severityLevel(), defect.speedLimitKmH());
        }

        System.out.println();
        System.out.println("--- Проверки ---");
        for (int creatorIndex = 0; creatorIndex < defectFactories.size(); creatorIndex++) {
            RailDefect defect = defectFactories.get(creatorIndex).createDefect(128.5, 45);
            String defectLabel = expectedCodes[creatorIndex];

            check("создатель " + expectedClassNames[creatorIndex] + " вернул нужный продукт",
                    defect.getClass().getSimpleName().equals(expectedClassNames[creatorIndex]));
            check("описание дефекта " + defectLabel + " не пустое и содержит код",
                    !defect.description().isEmpty() && defect.description().contains(defect.code()));
            check("характеристики дефекта " + defectLabel + " сохранены фабрикой",
                    defect.kmMark() == 128.5 && defect.lengthMm() == 45);
            check("оценка опасности дефекта " + defectLabel + " в допустимом диапазоне",
                    defect.severityLevel() >= 1 && defect.severityLevel() <= 4
                            && defect.speedLimitKmH() > 0);
        }

        // Четвёртый тип проверяется точно так же, как и остальные: через общие типы.
        RailDefect fourthDefect = defectFactories.get(3).createDefect(12.0, 80);
        check("четвёртый тип дефекта работает наравне с остальными",
                fourthDefect.code().equals("УК") && fourthDefect.title().contains("колеи")
                        && fourthDefect.lengthMm() == 80);

        finishChecks();
    }

    // --- служебное: проверки и завершение ---

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }

    static void finishChecks() {
        if (failedChecks > 0) {
            throw new AssertionError(" Factory Method: не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки Factory Method пройдены.");
    }
}

// --- Участник паттерна: продукт ---

/** Общий тип всех дефектов рельсов с описанием и измеримыми характеристиками. */
abstract class RailDefect {

    private final double kmMark;      // километровая отметка места дефекта
    private final int lengthMm;       // протяжённость дефекта, миллиметры

    RailDefect(double kmMark, int lengthMm) {
        this.kmMark = kmMark;
        this.lengthMm = lengthMm;
    }

    /** Код дефекта в журнале путевого хозяйства. */
    abstract String code();

    /** Человекочитаемое название дефекта. */
    abstract String title();

    /** Балл опасности по шкале от 1 до 4. */
    abstract int severityLevel();

    /** Ограничение скорости на участке, км/ч. */
    abstract int speedLimitKmH();

    final double kmMark() {
        return kmMark;
    }

    final int lengthMm() {
        return lengthMm;
    }

    /** Полное описание дефекта, используемое клиентским кодом. */
    final String description() {
        return title() + " [" + code() + "], пикет " + kmMark + " км, длина " + lengthMm
                + " мм, балл опасности " + severityLevel()
                + ", ограничение скорости " + speedLimitKmH() + " км/ч";
    }
}

// --- Участник паттерна: создатель ---

/** Абстрактная фабрика: фабричный метод возвращает только общий тип продукта. */
abstract class RailDefectFactory {

    abstract RailDefect createDefect(double kmMark, int lengthMm);
}

// --- Первые три продукта и их создатели ---

/** Головная трещина: наиболее опасный дефект головки рельса. */
class HeadCrackDefect extends RailDefect {

    HeadCrackDefect(double kmMark, int lengthMm) {
        super(kmMark, lengthMm);
    }

    String code() {
        return "ГТ";
    }

    String title() {
        return "Головная трещина рельса";
    }

    int severityLevel() {
        return 4;
    }

    int speedLimitKmH() {
        return 25;
    }
}

class HeadCrackFactory extends RailDefectFactory {

    RailDefect createDefect(double kmMark, int lengthMm) {
        return new HeadCrackDefect(kmMark, lengthMm);
    }
}

/** Трещина в шейке рельса: требует усиления и контроля. */
class WebCrackDefect extends RailDefect {

    WebCrackDefect(double kmMark, int lengthMm) {
        super(kmMark, lengthMm);
    }

    String code() {
        return "ТШ";
    }

    String title() {
        return "Трещина в шейке рельса";
    }

    int severityLevel() {
        return 3;
    }

    int speedLimitKmH() {
        return 40;
    }
}

class WebCrackFactory extends RailDefectFactory {

    RailDefect createDefect(double kmMark, int lengthMm) {
        return new WebCrackDefect(kmMark, lengthMm);
    }
}

/** Коррозия подошвы: развивается медленно, но снижает устойчивость рельса. */
class FootCorrosionDefect extends RailDefect {

    FootCorrosionDefect(double kmMark, int lengthMm) {
        super(kmMark, lengthMm);
    }

    String code() {
        return "КП";
    }

    String title() {
        return "Коррозия подошвы рельса";
    }

    int severityLevel() {
        return 2;
    }

    int speedLimitKmH() {
        return 60;
    }
}

class FootCorrosionFactory extends RailDefectFactory {

    RailDefect createDefect(double kmMark, int lengthMm) {
        return new FootCorrosionDefect(kmMark, lengthMm);
    }
}

// --- ЧЕТВЁРТЫЙ ТИП: добавлен новыми классами, прежние классы не изменялись ---

/** Уширение колеи: дефект пути, влияющий на сход с рельса. */
class GaugeWideningDefect extends RailDefect {

    GaugeWideningDefect(double kmMark, int lengthMm) {
        super(kmMark, lengthMm);
    }

    String code() {
        return "УК";
    }

    String title() {
        return "Уширение колеи";
    }

    int severityLevel() {
        return 3;
    }

    int speedLimitKmH() {
        return 50;
    }
}

class GaugeWideningFactory extends RailDefectFactory {

    RailDefect createDefect(double kmMark, int lengthMm) {
        return new GaugeWideningDefect(kmMark, lengthMm);
    }
}

Writing FactoryAfter.java


In [14]:
!java FactoryAfter.java

--- Дефекты рельсов, созданные фабричным методом ---
HeadCrackFactory Головная трещина рельса        код=ГТ  балл=4  ограничение=25 км/ч
WebCrackFactory Трещина в шейке рельса         код=ТШ  балл=3  ограничение=40 км/ч
FootCorrosionFactory Коррозия подошвы рельса        код=КП  балл=2  ограничение=60 км/ч
GaugeWideningFactory Уширение колеи                 код=УК  балл=3  ограничение=50 км/ч

--- Проверки ---
[OK]   создатель HeadCrackDefect вернул нужный продукт
[OK]   описание дефекта ГТ не пустое и содержит код
[OK]   характеристики дефекта ГТ сохранены фабрикой
[OK]   оценка опасности дефекта ГТ в допустимом диапазоне
[OK]   создатель WebCrackDefect вернул нужный продукт
[OK]   описание дефекта ТШ не пустое и содержит код
[OK]   характеристики дефекта ТШ сохранены фабрикой
[OK]   оценка опасности дефекта ТШ в допустимом диапазоне
[OK]   создатель FootCorrosionDefect вернул нужный продукт
[OK]   описание дефекта КП не пустое и содержит код
[OK]   характеристики дефекта КП сохранены

In [15]:
%%writefile factory.puml
@startuml
title Factory Method: дефекты рельсов (вариант 11, путевое хозяйство)

abstract class "RailDefect" as RailDefect {
  #kmMark : double
  #lengthMm : int
  +{abstract} code() : String
  +{abstract} title() : String
  +{abstract} severityLevel() : int
  +{abstract} speedLimitKmH() : int
  +description() : String
  +kmMark() : double
  +lengthMm() : int
}

abstract class "RailDefectFactory" as RailDefectFactory {
  +{abstract} createDefect(kmMark, lengthMm) : RailDefect
}

class HeadCrackDefect
class WebCrackDefect
class FootCorrosionDefect
class GaugeWideningDefect

class HeadCrackFactory
class WebCrackFactory
class FootCorrosionFactory
class GaugeWideningFactory

class FactoryAfter {
  +main(args) : void
  -check(checkName, passed) : void
  -finishChecks() : void
}

RailDefect <|-- HeadCrackDefect
RailDefect <|-- WebCrackDefect
RailDefect <|-- FootCorrosionDefect
RailDefect <|-- GaugeWideningDefect

RailDefectFactory <|-- HeadCrackFactory
RailDefectFactory <|-- WebCrackFactory
RailDefectFactory <|-- FootCorrosionFactory
RailDefectFactory <|-- GaugeWideningFactory

HeadCrackFactory ..> HeadCrackDefect : <<create>>
WebCrackFactory ..> WebCrackDefect : <<create>>
FootCorrosionFactory ..> FootCorrosionDefect : <<create>>
GaugeWideningFactory ..> GaugeWideningDefect : <<create>>

FactoryAfter o--> RailDefectFactory : использует
FactoryAfter ..> RailDefect : получает

note right of GaugeWideningDefect
  Четвёртый тип добавлен новыми классами,
  прежние классы не изменялись
end note
@enduml

Writing factory.puml


In [16]:
!plantuml -tpng factory.puml
from IPython.display import Image
Image("factory.png")

**Вывод по задаче 1 (Factory Method).** Создание через фабричный метод переносит решение «какой объект создать» из вызывающего кода в иерархию создателей: `main` работает только с `RailDefectFactory` и `RailDefect` и не знает конкретных классов дефектов. Четвёртый тип (уширение колеи) добавлен новыми классами `GaugeWideningDefect` и `GaugeWideningFactory` без правки прежних — это и есть открытость/закрытость. Фабрика излишня, когда тип один, создание тривиально и происходит в единственном месте: тогда `new` понятнее, чем дополнительная иерархия классов. Также фабрика не нужна, если конкретный тип фиксирован и никогда не расширяется.

---

## 4. Задача 2: Observer

**Сценарий.** Станция фиксирует прибытие поезда и должна оповестить получателей: табло, журнал, рассылку. В коде «до» станция сама знает про всех получателей.

### 4.1 Код «до» (запустить, не изменяя)

In [17]:
%%writefile ObserverBefore.java
// Код «до»: источник события жёстко связан со всеми получателями.
public class ObserverBefore {

    public static void main(String[] args) {
        Station station = new Station("Гомель");
        station.trainArrived("3022");
        station.trainArrived("6701");
    }
}

class Station {

    private final String stationName;

    Station(String stationName) {
        this.stationName = stationName;
    }

    void trainArrived(String trainNumber) {
        // Любой новый получатель требует правки этого метода
        System.out.println("[табло]  Поезд " + trainNumber + " прибыл на станцию " + stationName);
        System.out.println("[журнал] " + stationName + ": зафиксировано прибытие поезда " + trainNumber);
    }
}

Writing ObserverBefore.java


In [18]:
!java ObserverBefore.java

### 4.2 Задание

Переписать сценарий на **Observer** в файле `ObserverAfter.java`. Требования:

1. Интерфейс наблюдателя `ArrivalListener` с методом `onTrainArrived(String stationName, String trainNumber)`.
2. Субъект `Station` с методами `addListener`, `removeListener`, `trainArrived`; класс `Station` **не знает**, какие именно наблюдатели существуют.
3. Не менее трёх наблюдателей: табло, журнал, рассылка. Журнал должен **накапливать** записи, чтобы их можно было проверить.
4. Четвёртый наблюдатель (например, передача в АСУ) добавляется без правки класса `Station`.
5. Проверки в файле, минимум четыре:
   * после одного события журнал получил ровно одну запись;
   * после отписки наблюдателя его записи больше не появляются;
   * все подписчики получили одно и то же событие;
   * добавление четвёртого наблюдателя не потребовало изменения `Station`.
6. В выводе указать порядок получения события подписчиками и ответить на вопрос в выводе ниже.

**Подсказка:** список наблюдателей удобно хранить в `java.util.ArrayList`, обход — обычным `for`.

In [19]:
%%writefile ObserverAfter.java
// Задача 2, вариант 11 «Путевое хозяйство»: Observer для события «обнаружен дефект рельса».
// Субъект не знает конкретных наблюдателей: он работает только с интерфейсом DefectListener.
import java.util.ArrayList;
import java.util.Arrays;
import java.util.Collections;
import java.util.List;
import java.util.Objects;

public class ObserverAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {

        TrackSection trackSection = new TrackSection("Перегон Гомель — Жлобин, 128 км");
        RepairServiceListener repairService = new RepairServiceListener();
        DispatcherListener dispatcher = new DispatcherListener();
        DefectLogListener defectLog = new DefectLogListener();
        AsuTransmissionListener asuTransmission = new AsuTransmissionListener();

        // Подписка: субъект получает только объекты общего интерфейса.
        trackSection.addListener(repairService);
        trackSection.addListener(dispatcher);
        trackSection.addListener(defectLog);
        trackSection.addListener(asuTransmission);

        System.out.println("--- Первое событие: дефект обнаружен ---");
        trackSection.defectDetected("ГТ", 128.5, 4);
        List<String> firstDeliveryOrder = new ArrayList<>(trackSection.deliveryOrder());
        System.out.println("Порядок получения события: " + firstDeliveryOrder);

        DefectEvent expectedEvent = new DefectEvent(
                "Перегон Гомель — Жлобин, 128 км", "ГТ", 128.5, 4);

        System.out.println();
        System.out.println("--- Проверки первого события ---");
        check("все четыре подписчика получили событие",
                repairService.receivedCount() == 1 && dispatcher.receivedCount() == 1
                        && defectLog.messages().size() == 1 && asuTransmission.receivedCount() == 1);
        check("служба ремонта и диспетчер получили одно и то же событие",
                expectedEvent.equals(repairService.lastEvent())
                        && expectedEvent.equals(dispatcher.lastEvent()));
        check("журнал сохранил сообщение с кодом дефекта",
                defectLog.messages().size() == 1 && defectLog.messages().get(0).contains("ГТ"));
        check("четвёртый наблюдатель (АСУ) работает без правки TrackSection",
                expectedEvent.equals(asuTransmission.lastEvent())
                        && asuTransmission.name().contains("АСУ"));
        check("порядок уведомления совпадает с порядком подписки",
                firstDeliveryOrder.equals(Arrays.asList(
                        "Служба ремонта", "Диспетчер", "Журнал дефектов", "Передача в АСУ")));

        // Отписка диспетчера: после неё события ему больше не приходят.
        System.out.println();
        System.out.println("--- Отписка диспетчера и второе событие ---");
        trackSection.removeListener(dispatcher);
        check("количество подписчиков уменьшилось после отписки", trackSection.listenerCount() == 3);
        trackSection.defectDetected("ТШ", 129.2, 3);
        System.out.println("Порядок получения второго события: " + trackSection.deliveryOrder());

        System.out.println();
        System.out.println("--- Проверки после отписки ---");
        check("отписанный диспетчер не получил второе событие", dispatcher.receivedCount() == 1);
        check("активные подписчики получили второе событие",
                repairService.receivedCount() == 2 && defectLog.messages().size() == 2
                        && asuTransmission.receivedCount() == 2);
        check("журнал накопил две записи и сохранил новый код дефекта",
                defectLog.messages().size() == 2 && defectLog.messages().get(1).contains("ТШ"));

        // Повторная подписка показывает, что механизм регистрации обратим.
        trackSection.addListener(dispatcher);
        trackSection.defectDetected("КП", 130.0, 2);
        check("повторная подписка восстановила получение событий", dispatcher.receivedCount() == 2);

        finishChecks();
    }

    // --- служебное ---

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }

    static void finishChecks() {
        if (failedChecks > 0) {
            throw new AssertionError(" Observer: не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки Observer пройдены.");
    }
}

// --- Субъект: знает только интерфейс наблюдателя ---

final class DefectEvent {

    private final String sectionName;
    private final String defectCode;
    private final double kmMark;
    private final int severityLevel;

    DefectEvent(String sectionName, String defectCode, double kmMark, int severityLevel) {
        this.sectionName = sectionName;
        this.defectCode = defectCode;
        this.kmMark = kmMark;
        this.severityLevel = severityLevel;
    }

    String sectionName() {
        return sectionName;
    }

    String defectCode() {
        return defectCode;
    }

    double kmMark() {
        return kmMark;
    }

    int severityLevel() {
        return severityLevel;
    }

    String text() {
        return "код " + defectCode + ", пикет " + kmMark + " км, балл " + severityLevel
                + ", участок «" + sectionName + "»";
    }

    @Override
    public boolean equals(Object other) {
        if (!(other instanceof DefectEvent)) {
            return false;
        }
        DefectEvent otherEvent = (DefectEvent) other;
        return Double.compare(kmMark, otherEvent.kmMark) == 0
                && severityLevel == otherEvent.severityLevel
                && sectionName.equals(otherEvent.sectionName)
                && defectCode.equals(otherEvent.defectCode);
    }

    @Override
    public int hashCode() {
        return Objects.hash(sectionName, defectCode, kmMark, severityLevel);
    }
}

/** Интерфейс наблюдателя: метод обработки обнаруженного дефекта. */
interface DefectListener {

    void onDefectFound(DefectEvent event);

    /** Имя используется только для демонстрации порядка уведомления. */
    String name();
}

/** Субъект: участок пути, на котором обнаружен дефект. */
class TrackSection {

    private final String sectionName;
    private final List<DefectListener> listeners = new ArrayList<>();
    private final List<String> deliveryOrder = new ArrayList<>();

    TrackSection(String sectionName) {
        this.sectionName = sectionName;
    }

    void addListener(DefectListener listener) {
        listeners.add(listener);
    }

    void removeListener(DefectListener listener) {
        listeners.remove(listener);
    }

    /** Оповещает всех подписанных наблюдателей; о их конкретных типах ничего не знает. */
    void defectDetected(String defectCode, double kmMark, int severityLevel) {
        DefectEvent event = new DefectEvent(sectionName, defectCode, kmMark, severityLevel);
        deliveryOrder.clear();
        for (DefectListener listener : listeners) {
            listener.onDefectFound(event);
            deliveryOrder.add(listener.name());
        }
    }

    List<String> deliveryOrder() {
        return Collections.unmodifiableList(deliveryOrder);
    }

    int listenerCount() {
        return listeners.size();
    }
}

// --- Первые три наблюдателя ---

class RepairServiceListener implements DefectListener {

    private int receivedCount;
    private DefectEvent lastEvent;

    public void onDefectFound(DefectEvent event) {
        receivedCount++;
        lastEvent = event;
        System.out.println("[Служба ремонта]   Наряд-допуск на устранение: " + event.text());
    }

    public String name() {
        return "Служба ремонта";
    }

    int receivedCount() {
        return receivedCount;
    }

    DefectEvent lastEvent() {
        return lastEvent;
    }
}

class DispatcherListener implements DefectListener {

    private int receivedCount;
    private DefectEvent lastEvent;

    public void onDefectFound(DefectEvent event) {
        receivedCount++;
        lastEvent = event;
        System.out.println("[Диспетчер]        Ограничение скорости на участке: " + event.text());
    }

    public String name() {
        return "Диспетчер";
    }

    int receivedCount() {
        return receivedCount;
    }

    DefectEvent lastEvent() {
        return lastEvent;
    }
}

class DefectLogListener implements DefectListener {

    private final List<String> messages = new ArrayList<>();

    public void onDefectFound(DefectEvent event) {
        String message = "[Журнал дефектов] Записано: " + event.text();
        messages.add(message);
        System.out.println(message);
    }

    public String name() {
        return "Журнал дефектов";
    }

    List<String> messages() {
        return Collections.unmodifiableList(messages);
    }
}

// --- ЧЕТВЁРТЫЙ НАБЛЮДАТЕЛЬ: добавлен новым классом, TrackSection не изменялся ---

class AsuTransmissionListener implements DefectListener {

    private int receivedCount;
    private DefectEvent lastEvent;

    public void onDefectFound(DefectEvent event) {
        receivedCount++;
        lastEvent = event;
        System.out.println("[Передача в АСУ]   Пакет в автоматизированную систему: " + event.text());
    }

    public String name() {
        return "Передача в АСУ";
    }

    int receivedCount() {
        return receivedCount;
    }

    DefectEvent lastEvent() {
        return lastEvent;
    }
}

Writing ObserverAfter.java


In [20]:
!java ObserverAfter.java

--- Первое событие: дефект обнаружен ---
[Служба ремонта]   Наряд-допуск на устранение: код ГТ, пикет 128.5 км, балл 4, участок «Перегон Гомель — Жлобин, 128 км»
[Диспетчер]        Ограничение скорости на участке: код ГТ, пикет 128.5 км, балл 4, участок «Перегон Гомель — Жлобин, 128 км»
[Журнал дефектов] Записано: код ГТ, пикет 128.5 км, балл 4, участок «Перегон Гомель — Жлобин, 128 км»
[Передача в АСУ]   Пакет в автоматизированную систему: код ГТ, пикет 128.5 км, балл 4, участок «Перегон Гомель — Жлобин, 128 км»
Порядок получения события: [Служба ремонта, Диспетчер, Журнал дефектов, Передача в АСУ]

--- Проверки первого события ---
[OK]   все четыре подписчика получили событие
[OK]   служба ремонта и диспетчер получили одно и то же событие
[OK]   журнал сохранил сообщение с кодом дефекта
[OK]   четвёртый наблюдатель (АСУ) работает без правки TrackSection
[OK]   порядок уведомления совпадает с порядком подписки

--- Отписка диспетчера и второе событие ---
[OK]   количество подписчиков 

In [21]:
%%writefile observer.puml
@startuml
title Observer: событие «обнаружен дефект рельса» (вариант 11)

interface DefectListener {
  +onDefectFound(event) : void
  +name() : String
}

class DefectEvent {
  -sectionName : String
  -defectCode : String
  -kmMark : double
  -severityLevel : int
  +text() : String
}

class TrackSection {
  -sectionName : String
  -listeners : List<DefectListener>
  -deliveryOrder : List<String>
  +addListener(listener) : void
  +removeListener(listener) : void
  +defectDetected(defectCode, kmMark, severityLevel) : void
  +deliveryOrder() : List<String>
  +listenerCount() : int
}

class RepairServiceListener {
  -receivedCount : int
  -lastEvent : DefectEvent
}
class DispatcherListener {
  -receivedCount : int
  -lastEvent : DefectEvent
}
class DefectLogListener {
  -messages : List<String>
}
class AsuTransmissionListener {
  -receivedCount : int
  -lastEvent : DefectEvent
}

class ObserverAfter {
  +main(args) : void
}

DefectListener <|.. RepairServiceListener
DefectListener <|.. DispatcherListener
DefectListener <|.. DefectLogListener
DefectListener <|.. AsuTransmissionListener

TrackSection o--> DefectListener : уведомляет
TrackSection ..> DefectEvent : создаёт
DefectListener ..> DefectEvent : получает
ObserverAfter o--> TrackSection : подписывает / отписывает
ObserverAfter ..> DefectListener : использует

note right of AsuTransmissionListener
  Четвёртый наблюдатель подключён
  без изменения TrackSection
end note
@enduml

Writing observer.puml


In [22]:
!plantuml -tpng observer.puml
from IPython.display import Image
Image("observer.png")

**Вывод по задаче 2 (Observer).** Наблюдатель снял прямую зависимость `TrackSection` от получателей: субъект знает только интерфейс `DefectListener`, поэтому служба ремонта, диспетчер, журнал и система передачи в АСУ подключаются и отключаются без изменения субъекта. Порядок получения события в запуске: **Служба ремонта → Диспетчер → Журнал дефектов → Передача в АСУ**, то есть он совпадает с порядком подписки, но не является частью контракта и не должен использоваться бизнес-логикой. Новые проблемы паттерна: исключение в одном подписчике может прервать обход и лишить события остальных (нужна изоляция ошибок), забытая отписка оставляет субъекту ссылки на мёртвые объекты и ведёт к утечке памяти, а синхронное уведомление делает задержку одного наблюдателя задержкой всех.

## 5. Сравнение и выводы

### 5.1 Итоговая таблица

| Паттерн | Какую проблему кода «до» устраняет | Что добавляется без правки старого кода | Чем приходится заплатить | Где применим в моём проекте (вариант 11) |
|---|---|---|---|---|
| Strategy | Устраняет разбор варианта условным оператором внутри одного метода: у метода сразу две ответственности — выбрать алгоритм и его выполнить (нарушение OCP и SRP) | Новый алгоритм: новый класс стратегии; контекст `TariffCalculator` и старые стратегии не трогаются | Рост числа классов, необходимость выбирать и внедрять стратегию, более сложная конфигурация | Назначение вида работ по баллу опасности дефекта: «немедленный ремонт», «наблюдение», «плановая замена рельса» как взаимозаменяемые стратегии |
| Factory Method | Устраняет создание нужного типа через `if` и жёсткую связность клиента с конкретными классами продуктов | Новый тип дефекта рельса вместе с его создателем; `RailDefectFactory` и первые создатели не меняются | Дополнительная иерархия создателей, больше файлов, конкретный тип скрыт от клиента | Создание дефектов рельсов в журнале путевого хозяйства: головная трещина, трещина шейки, коррозия подошвы, уширение колеи |
| Observer | Устраняет жёсткую связь источника события со всеми получателями: субъект сам перечислял, кого оповещать | Новый наблюдатель (передача в АСУ) просто подписывается; `TrackSection` не правится | Непредсказуемый/неконтрактный порядок, ошибки в подписчиках, забытая отписка и утечки, синхронность | Событие «обнаружен дефект рельса»: служба ремонта, диспетчер, журнал регистрации, передача в АСУ |

### 5.2 Контрольные вопросы

1. **Какой принцип SOLID нарушает разбор типа условным оператором и как каждый из трёх паттернов его восстанавливает?**
   **Ответ:** прежде всего **OCP** (открыт для расширения, закрыт для изменения): новый вариант требует правки работающего метода; дополнительно страдает **SRP**, потому что метод и выбирает вариант, и считает/создаёт/оповещает. Strategy выносит алгоритмы в отдельные классы — контекст меняется только конфигурацией. Factory Method выносит выбор создаваемого типа в иерархию создателей — клиент зависит от абстрактного продукта. Observer добавляет получателей подпиской — источник зависит только от интерфейса наблюдателя (ещё и **DIP**: зависимость от абстракции, а не от конкретных классов).

2. **Чем Factory Method отличается от статического метода создания (`static factory method`) и от Abstract Factory?**
   **Ответ:** Factory Method — полиморфный метод абстрактного создателя, переопределяемый подклассами; каждый создатель даёт один продукт, новый продукт означает новый класс создателя. Статический фабричный метод — просто именованный `static`-метод внутри одного класса: он не переопределяется полиморфно и не образует иерархию, это лишь альтернатива конструктору. Abstract Factory инкапсулирует создание **семейства** взаимосвязанных продуктов через несколько фабричных методов (например, рельс + шпала + скрепление), тогда как Factory Method оперирует одним продуктом.

3. **Strategy и State имеют почти одинаковую структуру классов. В чём разница по назначению?**
   **Ответ:** Strategy — это взаимозаменяемость алгоритмов: клиент сам выбирает стратегию, стратегии не знают друг о друге и не меняют состояние контекста. State — это конечный автомат: контекст делегирует поведение текущему состоянию, состояние хранит переход и может переключить контекст в другое состояние; состояния часто знают друг о друге. По диаграмме они похожи, по смыслу Strategy отвечает «как сделать», State — «в каком состоянии система и что делать дальше».

4. **Почему Observer может привести к утечке памяти и как этого избежать?**
   **Ответ:** субъект держит сильные ссылки на всех подписчиков в своём списке. Если подписчик логически умер (окно закрылось, объект вышел из области видимости), но не вызвал `removeListener`, он остаётся достижимым через список долгоживущего субъекта и не может быть собран сборщиком мусора. Избегают явной отпиской в методе закрытия, `AutoCloseable`-обёрткой подписки (`try (Subscription s = ...)`), слабыми ссылками (`WeakReference`) в реестре слушателей и очисткой списка при уничтожении субъекта.

5. **Приведите пример, когда введение паттерна вредно. Что такое преждевременное усложнение?**
   **Ответ:** вредно, когда изменчивости нет: один-единственный тип дефекта, один получатель события или один фиксированный алгоритм. Тогда интерфейс, иерархия создателей или реестр слушателей добавляют файлы и косвенность, но не дают выигрыша — прямой `new` или прямой вызов читается лучше. Преждевременное усложнение — введение абстракций «на всякий случай», до того как появился второй реальный вариант использования; оно увеличивает число точек отказа и стоимость поддержки без практической пользы.

### 5.3 Три вывода по работе в целом

1. Все три паттерна заменяют хардкод-разбор типа и жёсткую связность полиморфизмом: поведение кода «до» сохраняется характеризующими тестами, а новое требование добавляется **новым классом**, без правки уже работающего кода.
2. Паттерны отвечают на разные вопросы: Strategy — взаимозаменяемый алгоритм (в варианте 11 назначение вида работ по дефекту), Factory Method — кто создаёт объект (тип дефекта рельса), Observer — кто должен узнать о событии (обнаружен дефект). На практике они легко сочетаются: фабрика создаёт дефект, наблюдатели оповещают о нём, стратегия назначает работы.
3. Цена паттернов — больше классов и косвенность, поэтому они оправданы лишь при реальной изменчивости. Самопроверка с `[OK]`/`[FAIL]`, три диаграммы и сравнительная таблица показывают, что рефакторинг прошёл без потери поведения и структура кода соответствует замыслу.

---

## 6. Варианты предметных областей

Номер варианта — по номеру студента в списке группы. Образец (раздел 2) выполняется всеми одинаково; задачи 1 и 2 — на своей предметной области.

| № | Предметная область | Фабрика создаёт | Стратегия варьирует | Наблюдатели получают событие |
|---|---|---|---|---|
| 1 | Грузовая станция | тип вагона | расчёт платы за пользование вагонами | вагон подан под погрузку |
| 2 | Локомотивное депо | вид ремонта (ТО-2, ТР-1, ТР-2) | приоритет заявок на ремонт | локомотив выдан из ремонта |
| 3 | Пассажирская касса | тип проездного документа | начисление скидки | рейс задержан |
| 4 | Контейнерный терминал | тип контейнера | выбор места на площадке | контейнер прибыл |
| 5 | Автотранспортное предприятие | тип транспортного средства | расчёт расхода топлива | путевой лист закрыт |
| 6 | Склад | тип тары | правило размещения (FIFO, ABC) | запас ниже нормы |
| 7 | Сортировочная станция | тип отцепа | очерёдность расформирования | состав расформирован |
| 8 | Пригородные перевозки | тип абонемента | тарификация по зонам | изменено расписание |
| 9 | Пункт таможенного оформления | тип декларации | выбор канала контроля | груз оформлен |
| 10 | Экспедиторская компания | тип заявки (авто, ж. д., море) | выбор перевозчика по критерию | статус отправки изменён |
| 11 | Путевое хозяйство | тип дефекта рельса | назначение вида работ | обнаружен дефект |
| 12 | Энергодиспетчерский пункт | тип потребителя | тарифный план (день, ночь) | авария на подстанции |
| 13 | Вагонное депо | тип неисправности | выбор отцепочного или безотцепочного ремонта | вагон забракован |
| 14 | Логистический центр | тип упаковки | правило консолидации отправок | сформирована отправка |
| 15 | Центр фирменного транспортного обслуживания | тип договора | расчёт скидки за объём | заявка согласована |

---

## 7. Чек-лист самопроверки

- [x] Раздел 1 выполнен, версия Java не ниже 11
- [x] `TariffBefore.java` и `TariffAfter.java` запущены, вывод сохранён
- [x] Отвечены вопросы к образцу (п. 2.5)
- [x] `FactoryAfter.java` написан, все проверки `[OK]`, в `main` нет разбора типов
- [x] Четвёртый тип вагона добавлен новым классом
- [x] `ObserverAfter.java` написан, все проверки `[OK]`, отписка работает
- [x] Четвёртый наблюдатель добавлен без правки класса `Station`
- [x] Построены три диаграммы: `strategy.puml`, `factory.puml`, `observer.puml`
- [x] Заполнена итоговая таблица (п. 5.1)
- [x] Отвечены пять контрольных вопросов и сформулированы три вывода
- [x] Блокнот выполняется сверху вниз без ошибок после *Среда выполнения → Перезапустить и выполнить все*
- [x] Запущена ячейка самопроверки, её вывод вставлен в описание pull request

## 8. Ячейка самопроверки

Ячейка сама компилирует и запускает все три файла, проверяет отсутствие `[FAIL]`, наличие диаграмм и минимальные признаки соблюдения требований. Запускать после того, как задачи выполнены.

In [23]:
#@title Запустить самопроверку и получить отчёт
import os, re, subprocess, datetime

ФИО_исполнителя = "Киселев И. Д."  #@param {type:"string"}
Группа_исполнителя = "ГИ-41"  #@param {type:"string"}
Номер_варианта = 11  #@param {type:"integer"}
ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = True  #@param {type:"boolean"}

ПРОГРАММЫ = [
    ("Образец, Strategy", "TariffAfter.java"),
    ("Задача 1, Factory Method", "FactoryAfter.java"),
    ("Задача 2, Observer", "ObserverAfter.java"),
]
ДИАГРАММЫ = ["strategy.puml", "factory.puml", "observer.puml"]

def запустить(имя_файла):
    """Запустить один исходный файл Java и вернуть (успех, пояснение)."""
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    if завершение.returncode != 0:
        значимые = [s for s in вывод.splitlines() if s.strip() and not s.startswith("Picked up")]
        первая_строка = значимые[0] if значимые else "без сообщения"
        return False, "аварийное завершение: " + первая_строка[:90]
    if "[FAIL]" in вывод:
        return False, "проверок не пройдено: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 3:
        return False, "проверок в файле только %d, требуется не менее 3" % успешных
    return True, "проверок пройдено: %d" % успешных

результаты = []
for название, файл in ПРОГРАММЫ:
    результаты.append((название, файл) + запустить(файл))

# Признаки соблюдения требований, которые можно проверить по тексту
замечания = []
def тело_метода(исходный_текст, сигнатура="main("):
    """Вернуть тело метода по сопоставлению фигурных скобок (без построчных комментариев)."""
    без_комментариев = re.sub(r"//[^\n]*", "", исходный_текст)
    начало = без_комментариев.find(сигнатура)
    if начало < 0:
        return ""
    открывающая = без_комментариев.find("{", начало)
    if открывающая < 0:
        return ""
    глубина = 0
    for позиция in range(открывающая, len(без_комментариев)):
        символ = без_комментариев[позиция]
        if символ == "{":
            глубина += 1
        elif символ == "}":
            глубина -= 1
            if глубина == 0:
                return без_комментариев[открывающая:позиция + 1]
    return без_комментариев[открывающая:]

if os.path.exists("FactoryAfter.java"):
    текст_фабрики = open("FactoryAfter.java", encoding="utf-8").read()
    блок_main = тело_метода(текст_фабрики)
    if re.search(r'\b(if|switch)\b', блок_main):
        замечания.append("в FactoryAfter.java метод main содержит ветвление по типу")
    if текст_фабрики.count("class ") < 5:
        замечания.append("в FactoryAfter.java меньше пяти классов: продукты, создатели и четвёртый тип, вероятно, не описаны")
if os.path.exists("ObserverAfter.java"):
    текст_наблюдателя = open("ObserverAfter.java", encoding="utf-8").read()
    for требуемое in ("addListener", "removeListener"):
        if требуемое not in текст_наблюдателя:
            замечания.append("в ObserverAfter.java нет метода %s" % требуемое)
отсутствующие_диаграммы = [d for d in ДИАГРАММЫ if not os.path.exists(d)]

строки = [
    "### Отчёт самопроверки «Песочница паттернов GoF»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s, вариант %d |" % (
        ФИО_исполнителя or "НЕ УКАЗАН", Группа_исполнителя or "НЕ УКАЗАНА", Номер_варианта),
    "| Дата и время отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "| Версия Java | %s |" % next(
        (s for s in subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()
         if "version" in s), "не определена"),
    "",
    "| Программа | Файл | Результат |",
    "|---|---|---|",
]
for название, файл, успех, пояснение in результаты:
    строки.append("| %s | `%s` | %s — %s |" % (название, файл, "пройдено" if успех else "НЕ ПРОЙДЕНО", пояснение))
строки += [
    "",
    "| Диаграммы | %s |" % ("все три на месте" if not отсутствующие_диаграммы
                            else "отсутствуют: " + ", ".join(отсутствующие_диаграммы)),
    "| Таблица и выводы | %s |" % ("заполнены" if ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
    "| Замечания | %s |" % ("нет" if not замечания else "; ".join(замечания)),
]
print("\n".join(строки))
print("\nСкопируйте отчёт выше в описание pull request.\n")

ошибки = []
if not ФИО_исполнителя.strip() or not Группа_исполнителя.strip():
    ошибки.append("не указаны ФИО и группа")
ошибки += ["%s: %s" % (название, пояснение) for название, файл, успех, пояснение in результаты if not успех]
if отсутствующие_диаграммы:
    ошибки.append("нет диаграмм: " + ", ".join(отсутствующие_диаграммы))
if not ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены итоговая таблица и выводы")
ошибки += замечания

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")

### Отчёт самопроверки «Песочница паттернов GoF»

| Параметр | Значение |
|---|---|
| Исполнитель | Киселев И. Д., группа ГИ-41, вариант 11 |
| Дата и время отчёта (UTC) | 2026-11-19 14:52 |
| Версия Java | openjdk version "21.0.13" 2026-10-20 LTS |

| Программа | Файл | Результат |
|---|---|---|
| Образец, Strategy | `TariffAfter.java` | пройдено — проверок пройдено: 5 |
| Задача 1, Factory Method | `FactoryAfter.java` | пройдено — проверок пройдено: 17 |
| Задача 2, Observer | `ObserverAfter.java` | пройдено — проверок пройдено: 10 |

| Диаграммы | все три на месте |
| Таблица и выводы | заполнены |
| Замечания | нет |

Скопируйте отчёт выше в описание pull request.

Самопроверка пройдена: работа готова к сдаче.


---

## 9. Критерии оценивания

| Критерий | Баллов |
|---|---|
| Образец запущен, вопросы п. 2.5 отвечены содержательно | 10 |
| Factory Method: корректная структура паттерна, в `main` нет разбора типов | 20 |
| Factory Method: четвёртый тип добавлен новым классом | 10 |
| Observer: корректная структура, работают подписка и отписка | 20 |
| Observer: четвёртый наблюдатель добавлен без правки субъекта | 10 |
| Проверки в каждом файле осмысленны (не `[OK]` ради `[OK]`) | 10 |
| Три диаграммы классов соответствуют коду | 10 |
| Итоговая таблица, контрольные вопросы, выводы | 10 |
| **Итого** | **100** |

Снижение: задача решена не на своём варианте — минус 20; код «после» повторяет разбор типа условным оператором внутри нового класса — минус 20; диаграмма не соответствует коду — минус 10.

## 10. Для желающих (сверх задания)

| Что сделать | Зачем |
|---|---|
| Подключить настоящий JUnit 5 вместо метода `check`: `!wget -q https://repo1.maven.org/maven2/org/junit/platform/junit-platform-console-standalone/1.10.2/junit-platform-console-standalone-1.10.2.jar` и запуск через `java -jar` | переход к инструменту, который используется в ЛР по модульному тестированию |
| Заменить три `if` в выборе стратегии на `java.util.Map` с ключом-перечислением (`enum`) | показать, что каталог стратегий сам может быть устранён из вызывающего кода |
| Реализовать Observer на `java.util.function.Consumer` вместо своего интерфейса | сравнить классический GoF и функциональный стиль |